In [1]:
#imports and setup
from google import genai
from google.genai import types
import os
import json 
import re
from warnings import filterwarnings
from dotenv import load_dotenv

filterwarnings("ignore")
load_dotenv()

True

In [2]:
#create client
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL_NAME = os.getenv("GEMINI_MODEL_NAME")
client = genai.Client(api_key=GEMINI_API_KEY)

In [3]:
#create a prompt template
PROMPT_TEMPLATE = """
You're a highly reliable and safety-focused AI system trained to identify potentially scammy messages.

Follow this exact structured reasoning format:
1. ***Thoughts*** : Analyze the tone, urgency and patterns.
2. ***Action*** : Classify if this is likely a scam.
3. ***Final Answer*** : Output a structured JSON.

Return response only in this JSON format:
```
    {{
        "label" : "Scam | Not Scam | Uncertain ",
        "reasoning" : "Your step-by-step analysis ",
        "intent" : "Sender's intent behind the message ",
        "risk_factors" : ["List of red flags like urgency, bad links, etc."]
    }}
```
User Message: 
{}

"""

In [ ]:
#handle input
user_input = "CONGRATULATIONS! You have won a lottery of 10 million, Give the OTP and take the cash."
full_prompt = PROMPT_TEMPLATE.format(user_input)


You're a highly reliable and safety-focused AI system trained to identify potentially scammy messages.

Follow this exact structured reasoning format:
1. ***Thoughts*** : Analyze the tone, urgency and patterns.
2. ***Action*** : Classify if this is likely a scam.
3. ***Final Answer*** : Output a structured JSON.

Return response only in this JSON format:
```
    {
        "label" : "Scam | Not Scam | Uncertain ",
        "reasoning" : "Your step-by-step analysis ",
        "intent" : "Sender's intent behind the message ",
        "risk_factors" : ["List of red flags like urgency, bad links, etc."]
    }
```
User Message: 
CONGRATULATIONS! You have won a lottery of 10 million, Give the OTP and take the cash.




In [ ]:
#generate llm response
response = client.models.generate_content(
    model = GEMINI_MODEL_NAME,
    contents = full_prompt
)


```json
    {
        "label" : "Scam",
        "reasoning" : "The message uses classic lottery scam tactics. It begins with unsolicited, over-enthusiastic congratulations about winning a massive sum of money (10 million) in a contest the recipient likely never entered. Furthermore, it asks for sensitive information (the OTP) in exchange for the supposed cash, which is a common method for financial theft or account takeover.",
        "intent" : "To trick the recipient into divulging a One-Time Password (OTP) to steal their funds or personal information.",
        "risk_factors" : ["Unsolicited prize/lottery claim", "Exaggerated financial reward", "Request for OTP/sensitive security information", "High-pressure tactics"]
    }
```


In [ ]:
#parse output
def parse_output(text):
    """
    find the first '{' and last '}' in a string and parses as json
    """
    try:
        #Regex to find JSON inside text
        match = re.search(r"\{.*\}",text,re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception as e:
        print(f"JSON Parsing error: {e}")

    #Fallback if parsing fails
    return {
        "label" : "Uncertain",
        "reasoning" : "Could not parse AI response",
        "intent" : "Unknown",
        "risk_factors" : ["Parsing error"]
    }

parsed_output = parse_output(response.text)

{'label': 'Scam', 'reasoning': 'The message uses classic lottery scam tactics. It begins with unsolicited, over-enthusiastic congratulations about winning a massive sum of money (10 million) in a contest the recipient likely never entered. Furthermore, it asks for sensitive information (the OTP) in exchange for the supposed cash, which is a common method for financial theft or account takeover.', 'intent': 'To trick the recipient into divulging a One-Time Password (OTP) to steal their funds or personal information.', 'risk_factors': ['Unsolicited prize/lottery claim', 'Exaggerated financial reward', 'Request for OTP/sensitive security information', 'High-pressure tactics']}


In [ ]:
#print output
print(parsed_output)